# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- Conduct EDA: visualization and statistical measures to systematically understand the structure of the data.
- Recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- Inspect NaNs, datatypes, and summary statistics

In [ ]:
#Load the data as aviation_df:
aviation_df = pd.read_csv("data/AviationData.csv", encoding = "cp1252")
aviation_df.head()

I can already see there are many differnet data types and NaNs... Let's see more info.

In [ ]:
aviation_df.info()

Oof, there's a lot of missing data in MANY columns... too many to even list here. Some columns look to have over 70,000 NaNs!

In [ ]:
aviation_df.isna()

Lots of missing data in the Latitude, Longitude, Airport.Code, Airport.Name, Air.carrier, and other columns.

In [ ]:
aviation_df.describe()

From just these statistics, it seems like the total number of fatal and serious injuries is 0 at least 75% of the time, but the max number is 349 fatalities, which is really bad. I can't tell yet if 349 is an outlier, but it looks good to notice that most of the time, there are 0 fatal injuries. Even serious and minor injuries are 0 at least 75% of the time.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- Inspect relevant columns
- Figure out any reasonable imputations
- Filter the dataset

In [ ]:
#See what values occur in the "Aircraft.Category" column:
aviation_df["Aircraft.Category"].value_counts()

In [ ]:
#We only want rows with "Airplane" as the entry:
aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
#Check that it worked:
aviation_df["Aircraft.Category"]

In [ ]:
#See what values occur in the "Amateur.Built" column:
aviation_df["Amateur.Built"].value_counts()

In [ ]:
#We only want rows with "No" as the entry because we only want professional
#builds:
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
#Check if it worked:
aviation_df["Amateur.Built"]

In [ ]:
#First, check what dtype the column "Event.Date" is:
aviation_df["Event.Date"].dtype

In [ ]:
#We don't want it to be an object, we want datetime, so let's convert it:
aviation_df["Event.Date"] = pd.to_datetime(aviation_df["Event.Date"])
#Let's check if that worked:
aviation_df["Event.Date"].dtype

In [ ]:
#Awesome, now we only want rows with dates from 1983 or later:
aviation_df = aviation_df[aviation_df["Event.Date"].dt.year >= 1983]
#Check if it worked:
aviation_df["Event.Date"]

### Cleaning and Constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct a metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious/fatal injury can be estimated as a fraction from this.

In [ ]:
#First, I need to see what columns are in the dataset:
aviation_df.columns

In [ ]:
#Maybe we can calculate how many total passengers were on each flight by adding
#together the "Total.Fatal.Injuries", "Total.Serious.Injuries",
#"Total.Minor.Injuries", and "Total.Uninjured" columns. Let's create a new
#column for this:
aviation_df["Total.Passengers"] = aviation_df[
    ["Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"]
    #Using .sum() ensures NaNs are skipped and axis = 1 ensures we are summing
    #across columns:
    ].sum(axis = 1)

#Check changes:
aviation_df.head()

In [ ]:
#For injured passengers of any time, NaN probably means 0 injuries recoreded,
#so let's replace NaNs in injury columns with 0s:
cols = [
    "Total.Passengers",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

aviation_df[cols] = aviation_df[cols].fillna(0)

In [ ]:
#Now let's calculate how many people on each flight had fatal or serious #injuries by calculating that fraction of total passengers on each flight:
aviation_df["Severe.or.Fatal.Fraction"] = (
    (aviation_df["Total.Fatal.Injuries"] +
     aviation_df["Total.Serious.Injuries"]) /
    aviation_df["Total.Passengers"]
)

#Check changes:
aviation_df.head()

In [ ]:
#Let's check if the "Severe.or.Fatal.Fraction" column has any NaNs (infinity
#after calculating that fraction:
print(aviation_df["Severe.or.Fatal.Fraction"].isna().sum())
aviation_df["Severe.or.Fatal.Fraction"].describe()

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized